# Explore the retail data

## Read the sheets
Check the shape and column names in both sheets.

In [ ]:
import pandas as pd
import numpy as np

sheets = pd.read_excel("data/raw/online_retail_II.xlsx", sheet_name=None)
for name, s in sheets.items():
    print(name, s.shape, list(s.columns))



## Join the sheets
The two years should give 1,067,371 rows and 8 columns.

In [ ]:
df = pd.concat(sheets.values(), ignore_index=True)
print(df.shape)
df.head()

## Fix text types
Keep invoice numbers and product codes as text. Trim spaces.

In [ ]:
print(df.dtypes)
for column in ["Invoice", "StockCode", "Description", "Country"]:
    df[column] = df[column].astype("string").str.strip()
df["StockCode"] = df["StockCode"].str.upper()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="raise")


## Save a quick copy
Use Parquet to avoid reading Excel again next time.

In [ ]:
from pathlib import Path

Path("data/processed").mkdir(parents=True, exist_ok=True)

df.to_parquet("data/processed/raw.parquet", index=False)

## Check the data
Missing values and unique values for each column.

In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(),
})

profile

## Save the profile

In [ ]:
profile.to_csv("data/processed/profile.csv")

## Apply the cleaning rules
Cleaning uses the same functions as build_matrix.py. Notebook 02 contains the steps. Training quantities are fixed before test-period returns are processed.


In [ ]:
from build_matrix import clean_data

cutoff = df["InvoiceDate"].max() - pd.DateOffset(months=2)
clean_df, cleaning_log, return_audit = clean_data(df, cutoff)
cleaning_log


## Check unusual codes
A different code format does not always mean a non-product.

In [ ]:
is_product_format = clean_df["StockCode"].str.fullmatch(r"\d{5}[A-Z]*", na=False)
unusual_codes = (
    clean_df.loc[~is_product_format, ["StockCode", "Description"]]
    .value_counts(dropna=False)
    .rename("rows")
    .reset_index()
)
unusual_codes.to_csv("data/processed/unusual_codes.csv", index=False)
unusual_codes.head(50)

## Save the cleaned data
PADS stays because its description names a product, but the low price is uncertain. SP1002 passes this rule; the later training support filter may remove it. See decision_log.md.


In [ ]:
clean_df.to_parquet("data/processed/clean.parquet", index=False)
cleaning_log.to_csv("data/processed/cleaning_log.csv", index=False)
return_summary = return_audit.groupby("period")[["return_units", "matched_units", "unmatched_units"]].sum()
return_summary


## Split by time
Use purchases before the cutoff for training. Keep later purchases for testing.

In [ ]:
train = clean_df.loc[clean_df["InvoiceDate"] < cutoff].copy()
test = clean_df.loc[clean_df["InvoiceDate"] >= cutoff].copy()

assert len(train) + len(test) == len(clean_df)
assert train["InvoiceDate"].max() < cutoff <= test["InvoiceDate"].min()

train.to_parquet("data/processed/train.parquet", index=False)
test.to_parquet("data/processed/test_transactions.parquet", index=False)
split_report = pd.DataFrame([
    {"set": "train", "rows": len(train), "start": train["InvoiceDate"].min(), "end": train["InvoiceDate"].max()},
    {"set": "test", "rows": len(test), "start": test["InvoiceDate"].min(), "end": test["InvoiceDate"].max()},
])
split_report.to_csv("data/processed/split_report.csv", index=False)
print("cutoff:", cutoff)
split_report